# 技能3 · Day 2 上机：实验设计与 A/B 测试统计

**版本**：v5.0 学习材料包
**配套**：notes.md（讲义）｜ data/README.md（真实数据集）｜ solution.ipynb（参考答案，做完再看）

## 学习目标
学完你能：
1. 在**真实 RCT 数据**（NSW 职业培训实验）上验证随机化均衡性，解释为什么 RCT 的均值差 = ATE
2. 为营销 A/B 测试计算所需样本量（给定基线转化率、MDE、α、power）
3. 完成比例 Z 检验 / t 检验、置信区间、事后功效分析，正确解读统计显著性
4. 用 CUPED 方差缩减技术提升 A/B 测试灵敏度

## 说明
本笔记本有 **6 个 TODO**，你需要自己填写代码。每个 TODO 有提示。
真实数据集：NSW 职业培训实验（真实 RCT），营销映射见下。

## 0. 环境准备
首次运行需安装依赖（取消注释执行一次）：

In [ ]:
# !pip install causaldata statsmodels scipy -q

## 1. 数据集背景与营销映射

**NSW 职业培训实验**：这是 Day 1 使用的同一数据集，但今天换一个视角--Day 1 把它当观测数据做后门调整，今天把它当**真实 RCT** 做 A/B 测试统计分析。NSW 本身就是随机对照试验（Randomized Controlled Trial），处理组接受了职业培训，对照组没有。

| NSW 变量 | 营销映射 | 角色 |
|---------|---------|------|
| `treat` | 是否看到新广告/收到优惠券 | 处理 T |
| `re78` | 转化率/GMV/客单价 | 结果 Y（连续） |
| `re78 > 0`（衍生） | 是否转化（0/1） | 结果 Y（二值，做比例检验） |
| `re75` | 实验前历史消费 | CUPED 协变量（前期数据） |
| `age`,`education`,... | 用户画像 | 协变量 X（均衡性检验） |

**核心对比**：Day 1 用观测对照（混杂严重），今天用实验对照（随机化消除混杂）-> 均值差就是因果效应！

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.api as sm
import statsmodels.stats.proportion as proportion
from causaldata import nsw
import warnings
warnings.filterwarnings('ignore')

## TODO 1-2：加载与验证 RCT 数据

In [ ]:
# TODO 1：加载真实 NSW RCT 数据
# 提示：nsw.load_pandas().data
# 要求：加载为 df，打印形状和前5行

# ===== 你的代码 =====
df = None  # TODO: 替换这行
# ====================

print(f"数据形状: {df.shape}")
df.head()

In [ ]:
# TODO 2：验证 RCT 均衡性 -- 检查处理组/对照组协变量是否均衡
# 提示：按 treat 分组，对比 age/education/re74/re75 等的均值
#       用 scipy.stats.ttest_ind 做逐变量均衡性检验（p>0.05 表示均衡）
# 要求：打印两组样本量 + 至少4个协变量的分组均值对比 + 逐变量 p 值

# ===== 你的代码 =====

# ====================

## 2. 为什么 RCT 中均值差 = ATE

**Day 1 的教训**：观测数据中，朴素均值差 = ATE + Bias（混杂偏差）。

**RCT 的数学保证**：随机化使处理组和对照组在所有特征（可观测 + 不可观测）上的期望分布相同：

$$E[Y(0)|T=1] = E[Y(0)|T=0] = E[Y(0)]$$

因此：

$$E[Y^{obs}|T=1] - E[Y^{obs}|T=0] = E[Y(1)] - E[Y(0)] = \text{ATE}$$

**随机化 = do 操作的物理实现**（Day 1 因果阶梯 L2）。不需要后门调整、不需要 DoWhy--简单的均值差就是无偏的 ATE 估计。

但 A/B 测试仍有统计挑战：**样本量够不够？功效足不足？显著性怎么判？** 这就是今天的核心。

## TODO 3：样本量计算

In [ ]:
# TODO 3：样本量计算
# 提示：n = (z_a/2 + z_b)^2 * [p1(1-p1) + p2(1-p2)] / (p2-p1)^2
#   z_a/2 = stats.norm.ppf(1-alpha/2), z_b = stats.norm.ppf(power)
#   p1 = 基线转化率, p2 = p1 + MDE
# 要求：写函数计算比例指标A/B测试每组样本量，计算基线5%/MDE=1%的场景

# ===== 你的代码 =====
def calculate_sample_size(baseline_rate, mde, alpha=0.05, power=0.80):
    pass  # TODO: 实现样本量公式

n_per_group = None  # TODO: 调用函数，基线0.05, MDE=0.01
# ====================

print(f"基线转化率: 5.0%")
print(f"最小可检测效应: +1.0个百分点 (到6.0%)")
print(f"每组所需样本量: {n_per_group:,}")
print(f"总样本量: {n_per_group * 2:,}")

## 3. 显著性检验与置信区间

### 假设检验框架

A/B 测试的核心是**假设检验**：
- **原假设 H0**：处理组与对照组无差异（效应 = 0）
- **备择假设 H1**：处理组优于对照组（效应 > 0，单侧）或 不等于 0（双侧）
- **p 值**：在 H0 为真时，观察到当前或更极端结果的概率
- **显著性水平 a**：通常 0.05 -- p < a 则拒绝 H0

**两类错误**：

| | H0 为真 | H0 为假 |
|---|---|---|
| 拒绝 H0 | **第一类错误（假阳性）**= a | 正确发现（功效 = 1-beta） |
| 不拒绝 H0 | 正确 | **第二类错误（假阴性）**= beta |

### 置信区间

95% 置信区间：如果重复实验 100 次，95 次的区间会包含真实效应值。区间不包含 0 等价于 p < 0.05。

### 营销 A/B 测试的四种场景

1. **广告创意 A/B**：新文案 vs 旧文案，比较 CTR（点击率）
2. **优惠券 A/B**：满减 vs 折扣，比较转化率
3. **落地页 A/B**：新布局 vs 旧布局，比较跳出率
4. **Push A/B**：个性化文案 vs 标准文案，比较打开率

## TODO 4：A/B 测试显著性检验

In [ ]:
# TODO 4：A/B 测试显著性检验
# 提示：
#   A) 连续指标 t 检验：stats.ttest_ind(treated_re78, control_re78)
#   B) 比例 Z 检验：proportion.proportions_ztest(
#        [treated_conversions, control_conversions], [n_treated, n_control])
#   先创建二值变量：df['employed'] = (df['re78'] > 0).astype(int)
# 要求：打印两种检验的统计量、p值、是否显著

# ===== 你的代码 =====
# A) t 检验 (re78)
t_stat = None  # TODO
p_val_t = None  # TODO

# B) 比例 Z 检验 (employed)
z_stat = None  # TODO
p_val_z = None  # TODO
# ====================

treated = df[df['treat']==1]['re78']
control = df[df['treat']==0]['re78']
print(f"=== t 检验 (re78) ===")
print(f"效应(均值差): {treated.mean() - control.mean():.2f}")
print(f"t={t_stat:.4f}, p={p_val_t:.4f}, 显著: {'是' if p_val_t < 0.05 else '否'}")
print(f"\n=== Z 检验 (employed) ===")
print(f"Z={z_stat:.4f}, p={p_val_z:.4f}, 显著: {'是' if p_val_z < 0.05 else '否'}")

## TODO 5：置信区间与事后功效分析

In [ ]:
# TODO 5：置信区间与事后功效分析
# 提示：
#   CI: diff +/- 1.96 * sqrt(var_t/n_t + var_c/n_c)
#   功效: sm.stats.TTestIndPower().power(effect_size, nobs1, alpha, ratio)
#   effect_size = |diff| / pooled_std  (Cohen's d)
# 要求：打印95%置信区间 + 事后功效 + 是否充分(>0.80)

# ===== 你的代码 =====
ci_lower = None  # TODO
ci_upper = None  # TODO
achieved_power = None  # TODO
# ====================

treated = df[df['treat']==1]['re78']
control = df[df['treat']==0]['re78']
diff = treated.mean() - control.mean()
print(f"效应(均值差): {diff:.2f}")
print(f"95%置信区间: [{ci_lower:.2f}, {ci_upper:.2f}]")
print(f"事后统计功效: {achieved_power:.4f}")
print(f"功效充分 (>0.80): {'是' if achieved_power > 0.80 else '否'}")

## 4. CUPED 方差缩减

### 为什么需要方差缩减

A/B 测试的灵敏度取决于**方差**：方差越大，标准误越大，t 值越小，越难检测到真实效应。

$$t = \frac{\text{效应}}{\text{标准误}} = \frac{\bar{Y}_t - \bar{Y}_c}{\sqrt{\text{Var}/n}}$$

**CUPED**（Controlled-Experiment Using Pre-Experiment Data，Deng et al. 2013）利用实验前的协变量 X 缩减 Y 的方差：

$$\theta = Y - \beta \cdot (X - \bar{X}), \quad \beta = \frac{\text{Cov}(Y, X)}{\text{Var}(X)}$$

方差缩减比例 = 1 - rho^2（rho 是 Y 与 X 的相关系数）。

**NSW 应用**：`re75`（1975 收入，实验前）与 `re78`（1978 收入，实验后）高度相关 -> 理想的 CUPED 协变量。

**营销应用**：用用户实验前的活跃度/消费数据作为 CUPED 协变量，在相同样本量下检测更小的效应 -> 直接节省实验成本。

## TODO 6（可选）：CUPED 方差缩减

In [ ]:
# TODO 6（可选）：CUPED 方差缩减
# 提示：
#   beta = np.cov(Y, X)[0,1] / np.var(X)
#   theta = Y - beta * (X - np.mean(X))  # CUPED 调整后的结果
#   Y = re78 (结果), X = re75 (前期协变量)
#   对比 CUPED 前后的方差、t 值、p 值
# 要求：打印方差缩减比例 + CUPED 前后 t 检验对比

# ===== 你的代码 =====
beta_cuped = None  # TODO
df['re78_cuped'] = None  # TODO
# ====================

# 对比检验
t_orig, p_orig = stats.ttest_ind(df[df['treat']==1]['re78'], df[df['treat']==0]['re78'])
t_cuped, p_cuped = stats.ttest_ind(df[df['treat']==1]['re78_cuped'], df[df['treat']==0]['re78_cuped'])

var_reduction = 1 - df['re78_cuped'].var() / df['re78'].var()
print(f"方差缩减: {var_reduction*100:.1f}%")
print(f"原始: t={t_orig:.4f}, p={p_orig:.4f}")
print(f"CUPED: t={t_cuped:.4f}, p={p_cuped:.4f}")

## 5. 准实验设计（DiD / RDD / ITS）

当你**无法随机化**时（成本太高、伦理问题、技术限制），准实验设计提供替代方案：

### 双重差分（DiD）

$$\text{DID} = (Y_{treatment, post} - Y_{treatment, pre}) - (Y_{control, post} - Y_{control, pre})$$

**关键假设**：平行趋势（无干预时两组变化趋势相同）
**营销场景**：A 城市上线 AI 推荐系统，B 城市没有，比较两城 GMV 变化差异

### 断点回归（RDD）

处理分配基于连续变量的阈值（如消费满 100 元发券）。在阈值附近，"刚好超过"和"刚好低于"近乎随机。
**营销场景**：CRM 对活跃度 >80 分的用户推送个性化内容，在 80 分附近做 RDD

### 中断时间序列（ITS）

$$Y_t = \beta_0 + \beta_1 \cdot \text{time} + \beta_2 \cdot \text{intervention} + \beta_3 \cdot \text{time} \times \text{intervention} + \epsilon$$

beta_2 是水平变化（干预瞬间的跳变），beta_3 是斜率变化（干预后趋势的改变）。
**营销场景**：全量上线新广告算法后，分析整体 ROI 时间序列趋势变化

> 深入阅读见 `reading.md` 的准实验设计条目（The Effect Ch.18 DiD / Ch.20 RDD）

## 6. 反思与前沿

### 反思问题
1. NSW 的 RCT 均衡性检验结果如何？与 Day 1 观测数据的不均衡对比，说明了什么？
2. 样本量计算告诉你什么？如果 NSW 的实际样本量不足，这对实验设计有什么启示？
3. CUPED 缩减了多少方差？t 值和 p 值如何变化？这对营销 A/B 测试的实践意义是什么？
4. 如果你的营销 A/B 测试中存在不依从（用户没看到实验版本），ITT 和 CACE 哪个更适合？

### 2026 前沿：CUPED 方差缩减
CUPED（Deng et al. 2013）是 A/B 测试的**工业标准方差缩减技术**，在微软/谷歌/亚马逊/Netflix 等公司广泛使用，2026 年仍是业界最佳实践。

**核心思想**：利用实验前的协变量 X（与结果 Y 相关但不受处理影响）调整 Y，缩减方差：

theta = Y - beta * (X - mean(X)),  beta = Cov(Y, X) / Var(X)

方差缩减比例 = 1 - rho^2（rho 是 Y 与 X 的相关系数）。在 NSW 中，re75（1975 收入）与 re78（1978 收入）高度相关，是理想的 CUPED 协变量。

**营销应用**：广告 A/B 测试中，用用户实验前的活跃度/消费/点击数据作为 CUPED 协变量，可在相同样本量下检测更小的效应，或用更少样本达到相同灵敏度 -> 直接节省实验成本。

参考：Deng, Xu, Kohavi, Walker (2013), WSDM. DOI: 10.1145/2433396.2433413

> 深入阅读见 `reading.md` 的 CUPED 条目。